# Lab W3D2 — Inference anatomy, by hand
Qwen2.5-1.5B-Instruct on a Colab T4. Hand-rolled TTFT/TPOT, KV cache measurement,
and static batching — before ever touching vLLM.


## Predict (by hand) — fill before running anything

1. **TTFT vs prompt length:** goes **up** (prefill reads the whole prompt before token 1).
2. **TPOT depends mostly on:** **model size and memory bandwidth** (decode is memory-bound, one token at a time).
3. **KV cache math (Qwen2.5-1.5B: 28 layers, 2 KV heads, head_dim 128, fp16):**
   - Per token: `2 (K,V) x 28 x 2 x 128 x 2 bytes = 57344 bytes = 28.0 KB/token`
   - At 4096 tokens: `4096 x 28 KB = 114688 KB / 1024 = 112 MB ≈ 0.109 GB`
4. **Static batching finishes when the** **slowest (longest) prompt in the batch** **finishes.**


## Cell 0 — pins + install (Cell A: profiling set only, no vLLM today)

In [ ]:
# Paste the pins + installer cell from ../shared/colab_scaffold.py first, e.g.:
#   TRANSFORMERS_PIN = "..."
#   ACCELERATE_PIN = "..."
#   def pip_install(*pkgs): ...
# Then INSTALL CELL A (the profiling set, same as day 1):

pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")


## Cell 2 — TTFT and TPOT by streaming

In [ ]:
import time, threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tok, skip_prompt=True,
                                    skip_special_tokens=True)
    kwargs = dict(**enc, max_new_tokens=new_tokens, do_sample=False,
                  streamer=streamer)
    th = threading.Thread(target=model.generate, kwargs=kwargs)
    t0 = time.time()
    th.start()
    stamps = []
    for _ in streamer:
        stamps.append(time.time())
    th.join()
    ttft = stamps[0] - t0
    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0
    total = stamps[-1] - t0
    return {"ttft_s": round(ttft, 4), "tpot_s": round(tpot, 4),
            "total_s": round(total, 4), "n_tokens": len(stamps)}

# Warm-up (not optional): first generation pays CUDA context init + kernel
# autotuning, which lands inside its TTFT. Throw one away.
measure_stream(prompt_of_len(128), new_tokens=8)

ttft_by_len = {}
for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(n, r)


## Cell 3 — KV growth vs. the formula

In [ ]:
import gc

def kv_formula_kb_per_token(layers=28, kv_heads=2, head_dim=128, dbytes=2):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024  # 28.0 KB

def cache_bytes(pkv):
    if hasattr(pkv, "key_cache"):
        tensors = list(pkv.key_cache) + list(pkv.value_cache)
    else:
        tensors = [t for layer in pkv for t in layer]
    return sum(t.numel() * t.element_size() for t in tensors)

def measure_kv(context: int, new_tokens: int = 256):
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()
    enc = tok(prompt_of_len(context), return_tensors="pt").to("cuda")
    before = torch.cuda.memory_allocated()
    out = model.generate(**enc, max_new_tokens=new_tokens, do_sample=False,
                         use_cache=True, return_dict_in_generate=True)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    total_tokens = out.sequences.shape[1]
    return {
        "context": context,
        "total_tokens": int(total_tokens),
        "peak_kb_per_token": round((peak - before) / total_tokens / 1024, 1),
        "kv_kb_per_token": round(cache_bytes(out.past_key_values) / total_tokens / 1024, 1),
    }

formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)
kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]
for r in kv_rows:
    print(r, "  vs formula", formula, "KB/token")

import json
with open("kv_check.json", "w") as f:
    json.dump({"formula_kb_per_token": formula,
               "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
               "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"]}, f)


**Read the two numbers against each other:**
- `kv_kb_per_token` should land on exactly **28.0** at every context length — that's the cache itself.
- `peak_kb_per_token` reads 2-3x higher and climbs with context — that's activations + allocator workspace during prefill, riding on top of the cache in the same measurement.
- "What must I budget per concurrent user?" → the cache number. "Will this request OOM the card?" → the peak number.


## Cell 4 — hand-rolled static batching

In [ ]:
# 24 requests: 18 that want 32 tokens, 6 that want 256. 2112 useful tokens.
QUEUE = [32, 32, 32, 256] * 6

def static_queue(batch: int, prompt: str = "Explain what an inference server does."):
    t0 = time.time(); useful = 0; slots = 0
    for i in range(0, len(QUEUE), batch):
        chunk = QUEUE[i:i + batch]
        n = max(chunk)
        enc = tok([prompt] * len(chunk), return_tensors="pt",
                  padding=True).to("cuda")
        model.generate(**enc, max_new_tokens=n, do_sample=False)
        useful += sum(chunk)
        slots += n * len(chunk)
    dt = time.time() - t0
    return {"batch": batch, "wall_s": round(dt, 2),
            "tokens_per_s": round(useful / dt, 1),
            "slot_efficiency": round(useful / slots, 3)}

batch_rows = {}
for n in [1, 4, 8]:
    r = static_queue(n)
    batch_rows[str(n)] = r["tokens_per_s"]
    print(r)


**What to expect:** throughput rises 1→8, but `slot_efficiency` collapses from 1.0
(batch 1) to roughly a third once lengths mix — short requests sit finished in a slot they
can't release, waiting on the 256-token member. That's the straggler tax continuous
batching removes tomorrow. Write your own 1-to-8 multiple on the card.


## Cell 5 — export baselines.json AND download it (non-negotiable)

In [ ]:
import json
baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {k: v for k, v in batch_rows.items()},
}
with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)
print(json.dumps(baselines, indent=2))


In [ ]:
from google.colab import files
files.download("baselines.json")


## Stretch — plot the per-token timestamps as a strip (optional)

In [ ]:
# Re-run measure_stream but keep `stamps` around, then:
import matplotlib.pyplot as plt
prompt = prompt_of_len(512)
enc = tok(prompt, return_tensors="pt").to("cuda")
from transformers import TextIteratorStreamer
streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
th = threading.Thread(target=model.generate, kwargs=dict(
    **enc, max_new_tokens=128, do_sample=False, streamer=streamer))
t0 = time.time(); th.start(); stamps = []
for _ in streamer:
    stamps.append(time.time() - t0)
th.join()

plt.figure(figsize=(8, 1.5))
plt.eventplot([stamps], orientation="horizontal")
plt.xlabel("seconds since request start")
plt.yticks([])
plt.title("Prefill gap (flat) then decode comb (steady)")
plt.tight_layout()
plt.show()


## Green check — verify_cell.py (paste as the last cell)

In [ ]:
# Green-check verifier for Lab W3D2 (inference anatomy).
# Paste this as the last cell of your day-2 notebook and run it. It reads
# baselines.json (the export you also downloaded) plus the KV measurement you
# saved to kv_check.json, and checks the schema and the sanity rules.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os

# Qwen2.5-1.5B KV cache: 2 x 28 layers x 2 kv_heads x 128 head_dim x 2 bytes.
KV_FORMULA_KB_PER_TOKEN = 2 * 28 * 2 * 128 * 2 / 1024  # 28.0


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def load_json(path: str):
    if not os.path.exists(path):
        fail(f"{path} not found")
    try:
        with open(path) as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")


def main() -> None:
    b = load_json("baselines.json")

    # schema
    for key in ("model", "dtype", "ttft_s", "tpot_s", "batch"):
        if key not in b:
            fail(f"baselines.json missing key: {key}")
    if not isinstance(b["ttft_s"], dict) or not b["ttft_s"]:
        fail("ttft_s must be a non-empty object keyed by prompt length")
    if not isinstance(b["batch"], dict):
        fail("batch must be an object keyed by batch size")
    for size in ("1", "4", "8"):
        if size not in b["batch"]:
            fail(f"batch missing size {size}")

    # sanity 1: TTFT rises with prompt length - the day's actual physics.
    tpot = b["tpot_s"]
    if not isinstance(tpot, (int, float)) or tpot <= 0:
        fail(f"tpot_s not a positive number: {tpot}")
    for plen, ttft in b["ttft_s"].items():
        if not isinstance(ttft, (int, float)) or ttft <= 0:
            fail(f"ttft_s[{plen}] not a positive number: {ttft}")
    if not b["ttft_s"]["2048"] > b["ttft_s"]["128"]:
        fail(f"TTFT did not rise with prompt length "
             f"(128: {b['ttft_s']['128']}, 2048: {b['ttft_s']['2048']}); "
             "prefill is not being measured")

    # sanity 2: batch-8 throughput beats batch-1
    b1, b8 = b["batch"]["1"], b["batch"]["8"]
    if not (isinstance(b1, (int, float)) and isinstance(b8, (int, float))):
        fail("batch tokens/s values must be numbers")
    if not b8 > b1:
        fail(f"batch-8 throughput ({b8}) not above batch-1 ({b1})")

    # sanity 3: measured KV within a factor of 2 of the formula
    kv = load_json("kv_check.json")
    measured = kv.get("measured_kb_per_token")
    if not isinstance(measured, (int, float)) or measured <= 0:
        fail("kv_check.json needs a positive measured_kb_per_token")
    lo, hi = KV_FORMULA_KB_PER_TOKEN / 2, KV_FORMULA_KB_PER_TOKEN * 2
    if not lo <= measured <= hi:
        fail(f"measured KV {measured} KB/token outside 2x of formula "
             f"{KV_FORMULA_KB_PER_TOKEN} (allowed {lo:.1f} to {hi:.1f})")

    print(f"ttft lengths: {sorted(b['ttft_s'])}, tpot_s: {tpot}")
    print(f"batch tokens/s 1/4/8: {b['batch']['1']}/{b['batch']['4']}/"
          f"{b['batch']['8']}")
    print(f"KV measured {measured} KB/token vs formula "
          f"{KV_FORMULA_KB_PER_TOKEN} KB/token")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)
